# Multi-Agent Collaborative Chat with AG2

This notebook demonstrates how to build multi-agent systems using [AG2](https://ag2.ai), an open-source framework for orchestrating multiple AI agents. AG2 enables agents powered by OpenAI models to collaborate, delegate, and solve complex tasks together.

**What you'll learn:**
- Setting up AG2 with OpenAI models
- Creating specialized agents with distinct roles
- Running a two-agent conversation
- Orchestrating a multi-agent GroupChat with automatic speaker selection

**Prerequisites:**
- OpenAI API key
- Python 3.10+

In [ ]:
%pip install "ag2[openai]>=0.11.4,<1.0" -q

In [ ]:
import os
from autogen import (
    AssistantAgent,
    UserProxyAgent,
    GroupChat,
    GroupChatManager,
    LLMConfig,
)

llm_config = LLMConfig({
    "model": "gpt-4o-mini",
    "api_key": os.environ["OPENAI_API_KEY"],
    "api_type": "openai",
})

## Part 1: Two-Agent Conversation

The simplest AG2 pattern: a user proxy sends a message to an assistant, which responds using an OpenAI model. The `UserProxyAgent` acts on behalf of the human, while the `AssistantAgent` is powered by GPT-4o-mini.

In [ ]:
assistant = AssistantAgent(
    name="Assistant",
    system_message=(
        "You are a helpful AI assistant. Provide clear, concise answers. "
        "Reply TERMINATE when the task is fully complete."
    ),
    llm_config=llm_config,
)

user_proxy = UserProxyAgent(
    name="User",
    human_input_mode="NEVER",
    max_consecutive_auto_reply=0,
    code_execution_config=False,
)

user_proxy.run(
    assistant,
    message="Explain the difference between fine-tuning and RAG in 3 bullet points.",
).process()

## Part 2: Multi-Agent GroupChat

AG2's `GroupChat` lets multiple specialized agents collaborate. A `GroupChatManager` powered by an OpenAI model selects which agent speaks next based on conversation context.

We'll create three agents:
- **Researcher** — gathers key facts
- **Writer** — crafts a polished summary
- **Critic** — reviews for accuracy and clarity

In [ ]:
researcher = AssistantAgent(
    name="Researcher",
    system_message=(
        "You are a research specialist. When given a topic, provide 3-5 "
        "key facts with sources. Be concise and structured."
    ),
    llm_config=llm_config,
)

writer = AssistantAgent(
    name="Writer",
    system_message=(
        "You are a skilled writer. Take the research provided and craft "
        "a clear, engaging summary in 2-3 paragraphs. Keep it under 200 words."
    ),
    llm_config=llm_config,
)

critic = AssistantAgent(
    name="Critic",
    system_message=(
        "You are a quality reviewer. Evaluate the content for accuracy, "
        "clarity, and completeness. If it meets standards, say APPROVED "
        "and reply TERMINATE. Otherwise give specific feedback."
    ),
    llm_config=llm_config,
)

In [ ]:
user_proxy_gc = UserProxyAgent(
    name="User",
    human_input_mode="NEVER",
    max_consecutive_auto_reply=0,
    code_execution_config=False,
)

group_chat = GroupChat(
    agents=[user_proxy_gc, researcher, writer, critic],
    messages=[],
    max_round=8,
    speaker_selection_method="auto",
)

manager = GroupChatManager(
    groupchat=group_chat,
    llm_config=llm_config,
)

user_proxy_gc.run(
    manager,
    message="Explain how OpenAI's function calling works and why it matters for building agents.",
).process()

## Next Steps

- **Tool Use**: Register Python functions as tools that agents can call — see [AG2 Tool Use guide](https://docs.ag2.ai/docs/tutorial/tool-use)
- **Code Execution**: Let agents write and run code in a sandboxed environment
- **Custom Speaker Selection**: Define explicit agent transitions with `allowed_or_disallowed_speaker_transitions`
- **Learn more**: [ag2.ai](https://ag2.ai) | [AG2 Documentation](https://docs.ag2.ai) | [GitHub](https://github.com/ag2ai/ag2)